This question involves the use of simple linear regression on the `Auto` dataset.

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from pathlib import Path

In [ ]:
# import dataset
cwd = Path().resolve()
auto_csv = cwd.parents[1] / "data" / "Auto.csv"
Auto = pd.read_csv(auto_csv, na_values=['?']).dropna().set_index("name")
Auto

In [ ]:
# response: mpg
# predictor: horsepower
X = pd.DataFrame({
    'intercept': np.ones(Auto.shape[0]),
    'horsepower': Auto['horsepower']
})
y = Auto['mpg']
model = sm.OLS(y, X)
result = model.fit()
result.summary()

In [ ]:
# RSE computation
print(np.sqrt(result.scale))
print(np.mean(Auto['mpg']))
print(np.sqrt(result.scale) / np.mean(Auto['mpg']))

In [ ]:
# confidence and prediction intervals on horsepower = 98
newX = pd.DataFrame({
    'intercept': [1],
    'horsepower': [98],
})
pred = result.get_prediction(newX)
pred.predicted_mean

In [ ]:
# confidence interval
pred.conf_int(alpha=0.05)

In [ ]:
# prediction interval
pred.conf_int(obs=True, alpha=0.05)

Questions
1. Is there a relationship between predictor and response?
    - The F statistic probability shows a small value < 0.05. Hence we can reject the null hypothesis that there is no relationship between the predictor and response and conclude that there is a relationship.
2. How strong is the relationship between predictor and response?
    - R^2 statistic shows that 0.6 of the variance of the response is explained by the predictor, showing a reasonably strong relationship.
    - In other words, approximately 60% of the variance in `mpg` is explained by `horsepower`.
    - RSE value is about 0.2 of the mean value of the response.
3. Is the relationship between predictor and response positive or negative?
    - Negative since coefficient is negative.
4. What is the predicted `mpg` associated with a `horsepower` of 98? What are the associated 95% confidence and prediction intervals?
    - The predicted `mpg` is 24.467
    - Confidence interval: [23.973, 24.961]
    - Prediction interval: [14.809, 34.124]

#### Regression Line Plot

In [ ]:
# plot of least squares regression line
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(X['horsepower'], y)
ax.plot(X['horsepower'], result.fittedvalues, c='r')

#### Diagnostic Plots

##### Residual-Fitted Values
Identify non-linearity patterns not modelled by the current model

In [ ]:
# Residual-Fitted Values
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(result.fittedvalues, result.get_influence().resid_studentized_internal)
ax.set_ylabel("studentized residual")
ax.set_xlabel("fitted values")
ax.axhline(y=0, c='k', ls='--')
plt.tight_layout()


##### Q-Q Plot
Check if residuals (as predicted by the model) follows normality assumptions. 

This is important for the various inferences we make from the regression, such as using t-tests, F-tests to determine statistical significance.

In [ ]:
# Q-Q Plot
fig, ax = plt.subplots(figsize=(5,5))
sm.qqplot(result.get_influence().resid_studentized_internal, line="45", ax=ax)
ax.set_ylabel("studentized residual quantiles")
ax.set_xlabel("theoretical quantiles")
plt.tight_layout()

# Q-Q Plot:
# - both tails above: left skew
# - both tails below: right skew
# - left tail above, right tail below: light tails
# - left tail below, right tail below: heavy tails

# in this case, it's close to normal with a slight left skew

##### Scale-Location Plot
Check if model follows homoskedasticity assumption (i.e. constant variance)

In [ ]:
# Scale-Location Plot

y = np.sqrt(np.abs(result.get_influence().resid_studentized_internal))
smooth = lowess(y, result.fittedvalues)
print(smooth.shape)
x_lowess, y_lowess = smooth[:, 0], smooth[:, 1]
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y=y, x=result.fittedvalues)
ax.plot(x_lowess, y_lowess, c='orange')
ax.set_xlabel("fitted values")
ax.set_ylabel("sqrt(abs(studentized residuals))")
plt.tight_layout()

# if homoscedasticity (constant variance of residuals) is there, then we should see a straight, horizontal line
# in this case, obviously, homoscedasticity does not hold.

# note: the lowess line shows how variance of residuals is changing


##### Cook's Distance Plot
Identify highly influential data points, these may be due to:
- data entry error
- valid but extreme observations
- a sign of missing predictors, non-linear relationships not being modelled.

In [ ]:
# Cook's Distance Plot i.e. Studentized Residual-Leverage Plot
fig, ax = plt.subplots(figsize=(6,6))
leverage = result.get_influence().hat_matrix_diag
resid_std = result.get_influence().resid_studentized_internal
num_params = 2 # simple linear regression
ax.scatter(leverage, resid_std)
ax.set_xlabel("leverage")
ax.set_ylabel("studentized residual")
ax.set_xlim(-0.001, 0.1)
ax.set_ylim(-3.5, 4.5)

# draw lines for Cook's distance = 1 and Cook's distance = 0.5
def plot_cooks_dist(d, p, resid_std, max_x = 0.2, **kwargs):
    y_min = np.min(resid_std)
    y_max = np.max(resid_std)

    x = np.linspace(0.001, max_x, 50)
    y1 = np.sqrt((d * p * (1 - x)) / x)
    l = ax.plot(x, y1, label=f"dist = {d}", **kwargs)

    x = np.linspace(0.001, max_x, 50)
    y2 = np.negative(np.sqrt((d * p * (1 - x)) / x))
    l = ax.plot(x, y2, **kwargs)
        
    

plot_cooks_dist(0.1, num_params, resid_std, c='b', ls='-.')
plot_cooks_dist(0.5, num_params, resid_std, c='r', ls='--')
plot_cooks_dist(1, num_params, resid_std, c='r', ls=':')
ax.legend(loc="lower right")

plt.tight_layout()
